<a href="https://colab.research.google.com/github/mikysetiawan/MachineLearningTerapan/blob/master/Tugas2_RecommendationArticle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "viniciuslambert/medium-data-science-articles-dataset",
    "medium-data-science-articles-2020.csv"
)

print("First 5 records:", df.head())

Using Colab cache for faster access to the 'medium-data-science-articles-dataset' dataset.
First 5 records:                                                  url  \
0  https://towardsdatascience.com/making-python-p...   
1  https://towardsdatascience.com/how-to-be-fancy...   
2  https://uxdesign.cc/how-exactly-do-you-find-in...   
3  https://towardsdatascience.com/from-scratch-to...   
4  https://www.cantorsparadise.com/the-waiting-pa...   

                                               title            author  \
0              Making Python Programs Blazingly Fast      martin.heinz   
1                        How to be fancy with Python           dipam44   
2  How exactly do you find insights from qualitat...   taylornguyen144   
3  From scratch to search: playing with your data...  stanislavprihoda   
4  The Waiting Paradox: An Intro to Probability D...        maikeelisa   

                                        author_page  \
0      https://towardsdatascience.com/@martin.heinz   


In [3]:
df.info()

print('Jumlah data artikel: ', len(df.url.unique()))
print('Jumlah data author: ', len(df.author.unique()))
print('Jumlah data tag: ', len(df.tag.unique()))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108021 entries, 0 to 108020
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   url           108021 non-null  object 
 1   title         108021 non-null  object 
 2   author        108021 non-null  object 
 3   author_page   108021 non-null  object 
 4   subtitle      38586 non-null   object 
 5   claps         108021 non-null  float64
 6   responses     108021 non-null  int64  
 7   reading_time  108021 non-null  int64  
 8   tag           108021 non-null  object 
 9   date          108021 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 8.2+ MB
Jumlah data artikel:  107971
Jumlah data author:  46809
Jumlah data tag:  7


In [4]:
# Mengecek missing value pada dataframe
df.isnull().sum()

,0
url,0
title,0
author,0
author_page,0
subtitle,69435
claps,0
responses,0
reading_time,0
tag,0
date,0


In [5]:
# Mengecek tag artikel yang unik
df.tag.unique()

array(['Data Science', 'Machine Learning', 'Artificial Inteligence',
       'Deep Learning', 'Data', 'Big Data', 'Analytics'], dtype=object)

# Model Development

In [6]:
# Inisialisasi TfidfVectorizer
tf = TfidfVectorizer()

# Melakukan perhitungan idf pada data tag
tf.fit(df['tag'])

# Mapping array dari fitur index integer ke fitur nama
tf.get_feature_names_out()

array(['analytics', 'artificial', 'big', 'data', 'deep', 'inteligence',
       'learning', 'machine', 'science'], dtype=object)

In [7]:
# Melakukan fit lalu ditransformasikan ke bentuk matrix
tfidf_matrix = tf.fit_transform(df['tag'])

# Melihat ukuran matrix tfidf
tfidf_matrix.shape

(108021, 9)

In [8]:
# Mengubah vektor tf-idf dalam bentuk matriks dengan fungsi todense()
tfidf_matrix.todense()

matrix([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.76076691],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.76076691],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.76076691],
        ...,
        [1.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [1.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [1.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ]])

In [9]:
# Membuat dataframe untuk melihat tf-idf matrix
# Kolom diisi dengan jenis masakan
# Baris diisi dengan nama resto

pd.DataFrame(
    tfidf_matrix.todense(),
    columns=tf.get_feature_names_out(),
    index=df.title
).sample(9, axis=1).sample(10, axis=0)

,artificial,big,analytics,deep,science,machine,inteligence,data,learning
title,,,,,,,,,
$1M spent on a predictive model/data science w/ $0 value and no user engagement?,0.000000,0.0,0.0,0.0,0.760767,0.00000,0.000000,0.649025,0.000000
物理與資料科學，只有一個想法的距離,0.000000,0.0,0.0,0.0,0.760767,0.00000,0.000000,0.649025,0.000000
"<strong class=""markup--strong markup--h3-strong"">Prophet в Машинном обучении простыми словами</strong>",0.000000,0.0,0.0,0.0,0.760767,0.00000,0.000000,0.649025,0.000000
How to Create a Populistic Bot and Why It Matters,0.707107,0.0,0.0,0.0,0.000000,0.00000,0.707107,0.000000,0.000000
Are We Losing The Battle For Data Literacy?,0.000000,0.0,0.0,0.0,0.000000,0.00000,0.000000,1.000000,0.000000
Iterator in Python,0.000000,0.0,0.0,0.0,0.760767,0.00000,0.000000,0.649025,0.000000
How to effectively manage annotation teams during COVID-19,0.000000,0.0,0.0,0.0,0.000000,0.72754,0.000000,0.000000,0.686065
Artificial Intelligence and 5G Are Here,0.707107,0.0,0.0,0.0,0.000000,0.00000,0.707107,0.000000,0.000000
Seamless Stitching of Perfect Labels,0.000000,0.0,0.0,0.0,0.000000,0.72754,0.000000,0.000000,0.686065


In [ ]:
# Menghitung cosine similarity pada matrix tf-idf
cosine_sim = cosine_similarity(tfidf_matrix)
cosine_sim

In [ ]:
# Membuat dataframe dari variabel cosine_sim dengan baris dan kolom berupa nama resto
cosine_sim_df = pd.DataFrame(cosine_sim, index=df['title'], columns=df['title'])
print('Shape:', cosine_sim_df.shape)

# Melihat similarity matrix pada setiap resto
cosine_sim_df.sample(5, axis=1).sample(10, axis=0)

In [ ]:
def article_recommendation(keyword, similarity_data=cosine_sim_df, items=df[['title', 'tag']], k=5):
    """
    Rekomendasi Resto berdasarkan kemiripan dataframe

    Parameter:
    ---
    nama_resto : tipe data string (str)
                Nama Restoran (index kemiripan dataframe)
    similarity_data : tipe data pd.DataFrame (object)
                      Kesamaan dataframe, simetrik, dengan resto sebagai
                      indeks dan kolom
    items : tipe data pd.DataFrame (object)
            Mengandung kedua nama dan fitur lainnya yang digunakan untuk mendefinisikan kemiripan
    k : tipe data integer (int)
        Banyaknya jumlah rekomendasi yang diberikan
    ---


    Pada index ini, kita mengambil k dengan nilai similarity terbesar
    pada index matrix yang diberikan (i).
    """


    # Mengambil data dengan menggunakan argpartition untuk melakukan partisi secara tidak langsung sepanjang sumbu yang diberikan
    # Dataframe diubah menjadi numpy
    # Range(start, stop, step)
    index = similarity_data.loc[:,keyword].to_numpy().argpartition(
        range(-1, -k, -1))

    # Mengambil data dengan similarity terbesar dari index yang ada
    closest = similarity_data.columns[index[-1:-(k+2):-1]]

    # Drop keyword agar nama resto yang dicari tidak muncul dalam daftar rekomendasi
    closest = closest.drop(keyword, errors='ignore')

    return pd.DataFrame(closest).merge(items).head(k)

In [ ]:
df[df.title.eq('machine learning')]

In [ ]:
# Mendapatkan rekomendasi restoran yang mirip dengan KFC
article_recommendation('machine learning')